# Text Feature Extraction for Vietnamese Real-Estate Listings

Turns the unstructured `name` / `description` fields of
[`tinixai/vietnam-real-estates`](https://huggingface.co/datasets/tinixai/vietnam-real-estates)
into model-ready features, and measures how much they add to a tabular price baseline.

**Three feature families**

1. **Domain keyword flags** (`kw_*`) — legal status (`sổ đỏ`, `sổ hồng`, `chính chủ`, `pháp lý rõ ràng`, `đang chờ sổ`), road accessibility (`ô tô đỗ cửa`, `mặt tiền`, `ngõ thông`, `xe hơi vào nhà`, `ngõ ba gác`) and condition/interior (`full nội thất`, `nội thất cơ bản`, `nhà mới`, `nhà cấp 4`). Matched diacritic-insensitively, with a negation guard so `không tranh chấp` does not read as a legal risk.
2. **Numeric entities** (`text_*`) — area, frontage × depth, floor count, bedrooms, bathrooms, road width and the advertised price, parsed out of free text and used to *impute* the heavily-sparse structured columns.
3. **Text representations** (`tfidf_*`, `phobert_*`) — TF-IDF + Truncated SVD by default, mean-pooled PhoBERT as an opt-in heavier path.

**Evaluation is out-of-time.** The parquet shards are chronologically ordered, so this trains on `shard_0000` (June 2025) and tests on `shard_0009` (March 2026) — a nine-month gap with no look-ahead.

Runtime: ~15 min on a Colab CPU runtime at the default sample sizes.

## 0. Setup

In [ ]:
%pip install -q lightgbm scikit-learn pandas pyarrow numpy

# Any ref containing real_estate/text works; the project branch is the default.
REF = "real-estate/baseline"

import os, subprocess, sys
if not os.path.isdir("uit-labs"):
    subprocess.run(["git", "clone", "-q", "-b", REF,
                    "https://github.com/vuongbinh/uit-labs.git"], check=True)
if os.path.abspath("uit-labs") not in sys.path:
    sys.path.insert(0, os.path.abspath("uit-labs"))


In [ ]:
import numpy as np
import pandas as pd

from real_estate.text import (
    ENTITY_COLUMNS,
    KeywordExtractor,
    TARGET_LEAKING_ENTITIES,
    TextFeatureConfig,
    TextFeaturePipeline,
    TfidfTextEncoder,
    clean_frame,
    extract_entities,
    load_shard_sample,
    parse_vn_number,
    prepare_for_tfidf,
    time_span,
)
from real_estate.text.benchmark import (
    BenchmarkConfig,
    entity_agreement_report,
    format_report,
    run_uplift_benchmark,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

## 1. Load a sample and check the data before modelling

`resolve_shard` downloads on first use and caches under `DATA_DIR`.

In [ ]:
DATA_DIR = "data"
TRAIN_SHARD, TEST_SHARD = "shard_0000", "shard_0009"

raw_train = load_shard_sample(TRAIN_SHARD, n_rows=60_000, seed=0, cache_dir=DATA_DIR)
raw_test = load_shard_sample(TEST_SHARD, n_rows=25_000, seed=1, cache_dir=DATA_DIR)
print("train", len(raw_train), time_span(raw_train))
print("test ", len(raw_test), time_span(raw_test))
raw_train.head(3)

### Three things the raw data does not tell you

Each of these was verified on the published shards and each one changes how you should model the target.

In [ ]:
import pyarrow.parquet as pq
from real_estate.text.data import resolve_shard

schema = pq.ParquetFile(resolve_shard(TRAIN_SHARD, cache_dir=DATA_DIR)).schema_arrow
print("1) `price` is declared float64 on the dataset card but is stored as", schema.field("price").type)
print("   parse_price() converts it; non-numeric values become NaN rather than a wrong number.")
print()

flags = KeywordExtractor().extract(raw_train)
rent = flags["kw_cho_thue"]
print(f"2) {rent.mean():.1%} of rows are rentals ('cho thuê'), mixed in with sales.")
print(f"   median price  rental = {raw_train.loc[rent, 'price'].median():,.0f} VND")
print(f"   median price  sale   = {raw_train.loc[~rent, 'price'].median():,.0f} VND")
print()

sparse = raw_train[["floor_count", "frontage_width", "house_depth", "road_width",
                    "bedroom_count", "bathroom_count"]].isna().mean().sort_values(ascending=False)
print("3) structured columns are heavily sparse — but the text usually states these values:")
print(sparse.map(lambda v: f"{v:.1%}").to_string())
print()
print(f"price range on this shard: {raw_train['price'].min():,.0f} .. {raw_train['price'].max():,.0f} VND")

## 2. Domain keyword flags

Patterns run against diacritic-folded text, so `"Ô Tô Đỗ Cửa"`, `"ô tô đỗ cửa"`, `"ôtô đỗ cổng"` and `"o to do cua"` all match the same rule.

The prevalence table below doubles as a lexicon health check: a rule that never fires on real data is dead weight, and the test suite fails if any pattern contains a literal diacritic (which could never match folded text).

In [ ]:
extractor = KeywordExtractor()
kw_train = extractor.extract(raw_train)
print(f"{kw_train.shape[1]} flags, {int((kw_train.sum(axis=1) == 0).mean() * 100)}% of rows match nothing\n")
extractor.prevalence(kw_train).head(28)

In [ ]:
# Spot-check the phrases named in the requirements.
demo = pd.DataFrame({"description": [
    "Bán nhà mặt tiền, sổ đỏ chính chủ, ô tô đỗ cửa, pháp lý rõ ràng",
    "Nhà trong ngõ ba gác, nội thất cơ bản, đang chờ sổ",
    "Không tranh chấp, không quy hoạch, xe hơi vào nhà, full nội thất",
    "Nhà cấp 4 cũ, hẻm nhỏ",
]})
cols = ["kw_so_do", "kw_chinh_chu", "kw_mat_tien", "kw_o_to_do_cua", "kw_ngo_ba_gac",
        "kw_xe_hoi_vao_nha", "kw_noi_that_co_ban", "kw_dang_cho_so", "kw_nha_cap_4",
        "kw_tranh_chap", "kw_quy_hoach", "kw_khong_quy_hoach"]
KeywordExtractor().extract(demo)[cols].assign(description=demo["description"])

Row 3 is the reason the negation guard exists: `không tranh chấp` / `không quy hoạch` advertise the *absence* of an encumbrance, so `kw_tranh_chap` and `kw_quy_hoach` stay `False` while `kw_khong_quy_hoach` turns on.

## 3. Numeric entities and text-driven imputation

`extract_entities` parses the numbers listings state in prose. Because the structured columns are sparse, most of the value here is **recovery**: filling `house_depth`, `floor_count` and `road_width` where the source record has nothing.

In [ ]:
ent_train = extract_entities(raw_train)
ent_train.describe().T[["count", "mean", "50%"]]

In [ ]:
# Validation: where the structured column and the parsed entity both exist, do they agree?
entity_agreement_report(raw_train, ent_train)

Reading the table: `text_coverage` is how often the text yields a value, `col missing` is how often the structured column is empty, `agreement` is exact-match rate where both exist, and `recovered` is the number of previously-null cells the text fills.

`N lầu` is deliberately kept in its own `text_lau_count` column instead of being added to `floor_count`: southern listings use it both for "levels above ground" and "levels total", so converting it would bake in an unfounded assumption.

In [ ]:
# Vietnamese number parsing: both separators, and 3-digit groups read as thousands.
for token in ["4", "7.45", "1,3", "1.300", "1.300.000"]:
    print(f"  {token:>12}  ->  {parse_vn_number(token):,.2f}")

## 4. Why `text_price_vnd` is measured but never modelled

The advertised price is written into the listing itself (`"giá: 7.45 tỷ"`), so parsing it back out and feeding it to a model trained on `price` is circular — it scores transcription, not valuation. In a first run this single column took **57% of total split gain** and flattered the whole entities arm by ~26% RMSLE.

Two controls are in place:

- `TARGET_LEAKING_ENTITIES` is excluded from every model feature block.
- Price mentions are **redacted from the TF-IDF/PhoBERT input** too, otherwise digit n-grams smuggle the same information back in.

The cell below is the direct check: two identical listings quoting different prices must encode *identically* once redaction is on.

In [ ]:
BASE = "Nhà mặt tiền sổ đỏ chính chủ, ô tô đỗ cửa, diện tích 40m2, 3 phòng ngủ"
print("redacted :", prepare_for_tfidf(f"{BASE}, giá 5.5 tỷ"))
print("unredacted:", prepare_for_tfidf(f"{BASE}, giá 5.5 tỷ", redact_price=False))

demo_frame = pd.DataFrame({"name": ["A", "B"], "description": [
    f"{BASE}, giá 5.5 tỷ", f"{BASE}, giá 7.25 tỷ"]})
for redact in (True, False):
    cfg = TextFeatureConfig(use_keywords=False, use_entities=False,
                            tfidf_components=2, tfidf_min_df=1, redact_price=redact)
    vec = TextFeaturePipeline(cfg).fit_transform(demo_frame).to_numpy(dtype=float)
    identical = np.allclose(vec[0], vec[1], atol=1e-6)
    print(f"redact_price={redact!s:5} -> price-only difference visible to TF-IDF: {not identical}")

## 5. TF-IDF representation

Vietnamese is syllable-delimited, so word **bigrams** are what capture domain phrases like `sổ đỏ` and `mặt tiền`. The character view is a robustness alternative; §7 compares them.

The encoder is fitted on training text only and reused unchanged on the test period.

In [ ]:
corpus = [prepare_for_tfidf(t) for t in
          (raw_train["name"].fillna("") + " " + raw_train["description"].fillna(""))]

tfidf = TfidfTextEncoder(mode="word", n_components=32, min_df=10).fit(corpus[:20_000])
print("vocabulary:", tfidf.vocabulary_sizes)
print("explained variance:", {k: round(v, 3) for k, v in tfidf.explained_variance.items()})

for component in (0, 1):
    terms = ", ".join(t for t, _ in tfidf.top_terms("w", component, k=10))
    print(f"\ncomponent {component}: {terms}")

## 6. Marginal uplift benchmark

One reference tabular block (numerics, ratios, frequency + smoothed target encodings of geography) trained with LightGBM on `log1p(price)`, then the same block plus each text family in turn. Every arm is a superset of the previous one, so each row of the delta table is a **marginal** gain.

Early stopping uses the latest 15% of the *training* period, never the test period. Two seeds per arm, because an uplift smaller than the seed-to-seed spread is not a finding.

The tabular block here is a reference harness, not the project baseline — `run_uplift_benchmark` accepts a pre-built matrix so the HYPE-12 pipeline can be dropped in.

In [ ]:
config = BenchmarkConfig(
    n_train=60_000, n_test=25_000, n_seeds=2,
    tfidf_mode="word", tfidf_components=128, tfidf_min_df=5,
    drop_rentals=True, exclude_target_leaking_entities=True, redact_price_in_text=True,
)
report = run_uplift_benchmark(TRAIN_SHARD, TEST_SHARD, config=config, cache_dir=DATA_DIR)
print(format_report(report))

## 7. Comparing TF-IDF analyzers

Word bigrams against character 3–5-grams against both concatenated, at matched sample sizes.

In [ ]:
rows = []
for mode in ("word", "char", "both"):
    cfg = BenchmarkConfig(n_train=40_000, n_test=20_000, n_seeds=1,
                          tfidf_mode=mode, tfidf_components=128)
    rep = run_uplift_benchmark(TRAIN_SHARD, TEST_SHARD, config=cfg, cache_dir=DATA_DIR)
    for arm in ("tabular", "+tfidf"):
        rows.append({"mode": mode, "arm": arm, **rep["arms"][arm]["metrics"]})

comp = pd.DataFrame(rows).drop(columns=["n"])
comp

In [ ]:
gain = (comp[comp.arm == "+tfidf"].set_index("mode")["rmsle"]
        - comp[comp.arm == "tabular"].set_index("mode")["rmsle"])
print("RMSLE change from adding TF-IDF, by analyzer (negative is better):")
print(gain.map(lambda v: f"{v:+.4f}").to_string())

## 8. Optional: PhoBERT embeddings

Heavier and opt-in. `torch`/`transformers` are imported lazily, so nothing above needs them. Mean-pooled `vinai/phobert-base`, reduced with SVD, on a subsample — CPU encoding is the bottleneck.

Diacritics are preserved on this path (unlike TF-IDF) because PhoBERT's vocabulary distinguishes them; price mentions are still redacted.

In [ ]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q transformers

cfg = BenchmarkConfig(
    n_train=4_000, n_test=2_000, n_seeds=1,
    tfidf_mode="word", tfidf_components=128, use_phobert=True,
    phobert={"n_components": 64, "max_length": 128, "batch_size": 16},
)
phobert_report = run_uplift_benchmark(TRAIN_SHARD, TEST_SHARD, config=cfg, cache_dir=DATA_DIR)
print(format_report(phobert_report))

## 9. Using these features in the baseline pipeline

Fit the pipeline on training rows only, transform both sides, and column-bind onto the tabular block. The index contract is what keeps this safe: `transform` returns rows aligned to the input frame, and imputation must happen **before** tabular encoding so derived ratios use the recovered numbers.

In [ ]:
from real_estate.text import NUMERIC_COLUMNS, impute_from_text
from real_estate.text.benchmark import TabularEncoder

# Cleaned frames: rentals removed via the keyword flag (see section 1).
kw_train = KeywordExtractor().extract(raw_train)
kw_test = KeywordExtractor().extract(raw_test)
train_df = clean_frame(raw_train, drop_mask=kw_train["kw_cho_thue"])
test_df = clean_frame(raw_test, drop_mask=kw_test["kw_cho_thue"])
print("clean rows:", len(train_df), len(test_df))

pipeline = TextFeaturePipeline(TextFeatureConfig(
    tfidf_mode="word", tfidf_components=128, tfidf_min_df=5,
    redact_price=True,          # keep the regression target out of the text features
))
train_features = pipeline.fit_transform(train_df)   # fits TF-IDF/SVD on TRAIN only
test_features = pipeline.transform(test_df)         # reuses that fitted state

# 1. Impute the sparse structured columns from text, THEN encode the tabular
#    block, so derived ratios use the recovered numbers rather than the nulls.
numeric = [c for c in NUMERIC_COLUMNS if c in train_df.columns]
train_imp, test_imp = train_df.copy(), test_df.copy()
train_imp[numeric] = impute_from_text(train_df[numeric], train_features[ENTITY_COLUMNS])[numeric]
test_imp[numeric] = impute_from_text(test_df[numeric], test_features[ENTITY_COLUMNS])[numeric]

encoder = TabularEncoder().fit(train_imp)
tabular_train = encoder.transform(train_imp)
tabular_test = encoder.transform(test_imp)

# 2. Bind the text features on, minus the target-restating entity.
drop = [c for c in TARGET_LEAKING_ENTITIES if c in train_features.columns]
X_train = pd.concat([tabular_train, train_features.drop(columns=drop)], axis=1)
X_test = pd.concat([tabular_test, test_features.drop(columns=drop)], axis=1)

assert list(X_train.columns) == list(X_test.columns)
assert len(X_train) == len(train_df) and len(X_test) == len(test_df)
print("tabular:", tabular_train.shape[1], "-> with text:", X_train.shape[1], "features")


## 10. Takeaways

- **The biggest win is entity recovery, not bag-of-words.** `house_depth` is null in ~96% of rows and `floor_count` in ~84%, yet the text states them constantly; parsed values agree with the structured column in ~88–93% of overlapping rows.
- **Keyword flags are cheap and additive**, but individually weak: most carry a small marginal gain because the geographic target encodings already absorb much of the same signal.
- **TF-IDF adds real signal on top**, and it survives an honest out-of-time test.
- **Two dataset traps** must be handled before any baseline is trustworthy: `price` is a *string* column despite the card declaring `float64`, and ~17–19% of rows are *rentals* whose monthly asking price is orders of magnitude below a sale price. `kw_cho_thue` is how they get separated.
- **Leakage is the main hazard of text features here.** The asking price is in the prose; redact it or the benchmark measures nothing.